In [1]:
# Consolidated imports and global configuration
import os
import json
import re
import pandas as pd
import numpy as np
import networkx as nx
import chromadb
from pathlib import Path
from collections import Counter
from chromadb.utils import embedding_functions
from sentence_transformers import SentenceTransformer, util
from llama_cpp import Llama, LlamaGrammar

# Constants / paths / globals
RESULTS_DIR = Path('../data/results')
ATTCK_DIR   = Path('../data/attck')
CHROMA_DIR  = Path('../data/chroma')
MODEL_PATH  = '../models/qwen2.5-3b-instruct-q4_k_m.gguf'
RANDOM_SEED = 42

# Ensure folders exist
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CHROMA_DIR.mkdir(parents=True, exist_ok=True)

print('Top-level imports consolidated. Use these across the notebook.')

Top-level imports consolidated. Use these across the notebook.


In [2]:
# Data file discovery (assumes top-level imports already run)
# Path to the CICIDS data directory
data_path = '../data/cicids'
# List all CSV files in the directory
csv_files = [f for f in os.listdir(data_path) if f.endswith('.csv')]
# Filter out Monday and Wednesday files
csv_files_filtered = [f for f in csv_files if ('Monday' not in f and 'Wednesday' not in f)]

print("Files to read (excluding Monday/Wednesday):")
for f in csv_files_filtered:
    print(f"  - {f}")

# Load and concatenate below (actual reading happens later after imports)

Files to read (excluding Monday/Wednesday):
  - Tuesday-WorkingHours.pcap_ISCX.csv
  - Friday-WorkingHours-Morning.pcap_ISCX.csv
  - Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
  - Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
  - Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
  - Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv


In [3]:
# Combine all CSV files into one dataframe
dfs = []
for f in csv_files_filtered:
    file_path = os.path.join(data_path, f)
    df_temp = pd.read_csv(file_path)
    # Add source file column for reference
    df_temp['Source_File'] = f
    dfs.append(df_temp)
    print(f"Loaded {f}: {len(df_temp)} rows")

# Concatenate all dataframes
df_combined = pd.concat(dfs, ignore_index=True)

print(f"\nTotal combined rows: {len(df_combined)}")
print(f"Total columns: {len(df_combined.columns)}")

Loaded Tuesday-WorkingHours.pcap_ISCX.csv: 445909 rows
Loaded Friday-WorkingHours-Morning.pcap_ISCX.csv: 191033 rows
Loaded Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: 288602 rows
Loaded Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: 225745 rows
Loaded Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: 170366 rows
Loaded Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: 286467 rows

Total combined rows: 1608122
Total columns: 80


In [4]:
df_combined.columns

Index([' Destination Port', ' Flow Duration', ' Total Fwd Packets',
       ' Total Backward Packets', 'Total Length of Fwd Packets',
       ' Total Length of Bwd Packets', ' Fwd Packet Length Max',
       ' Fwd Packet Length Min', ' Fwd Packet Length Mean',
       ' Fwd Packet Length Std', 'Bwd Packet Length Max',
       ' Bwd Packet Length Min', ' Bwd Packet Length Mean',
       ' Bwd Packet Length Std', 'Flow Bytes/s', ' Flow Packets/s',
       ' Flow IAT Mean', ' Flow IAT Std', ' Flow IAT Max', ' Flow IAT Min',
       'Fwd IAT Total', ' Fwd IAT Mean', ' Fwd IAT Std', ' Fwd IAT Max',
       ' Fwd IAT Min', 'Bwd IAT Total', ' Bwd IAT Mean', ' Bwd IAT Std',
       ' Bwd IAT Max', ' Bwd IAT Min', 'Fwd PSH Flags', ' Bwd PSH Flags',
       ' Fwd URG Flags', ' Bwd URG Flags', ' Fwd Header Length',
       ' Bwd Header Length', 'Fwd Packets/s', ' Bwd Packets/s',
       ' Min Packet Length', ' Max Packet Length', ' Packet Length Mean',
       ' Packet Length Std', ' Packet Length Variance', '

In [5]:
print("\nLabel Counts:")
print(df_combined[' Label'].value_counts())


Label Counts:
 Label
BENIGN                        1303148
PortScan                       158930
DDoS                           128027
FTP-Patator                      7938
SSH-Patator                      5897
Bot                              1966
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Name: count, dtype: int64


In [6]:
# Clean the ' Label' column
df_combined[' Label'] = df_combined[' Label'].str.replace('�', '-', regex=False)

# Replace multiple spaces/hyphens with single hyphen and ensure spaces around it
df_combined[' Label'] = df_combined[' Label'].str.replace(r'\s*-\s*', ' - ', regex=True)

# Strip extra whitespace
df_combined[' Label'] = df_combined[' Label'].str.strip()

# Check the cleaned labels
print("Cleaned Label Counts:")
print(df_combined[' Label'].value_counts())

Cleaned Label Counts:
 Label
BENIGN                        1303148
PortScan                       158930
DDoS                           128027
FTP - Patator                    7938
SSH - Patator                    5897
Bot                              1966
Web Attack - Brute Force         1507
Web Attack - XSS                  652
Infiltration                       36
Web Attack - Sql Injection         21
Name: count, dtype: int64


In [7]:
# Stratified sampling - get up to 2000 samples per label (if available)
if 'df_combined' not in globals():
    raise RuntimeError('`df_combined` not found. Run the data-loading cells first.')
sampled_dfs = []

for label in df_combined[' Label'].unique():
    label_df = df_combined[df_combined[' Label'] == label]
    n_samples = min(2000, len(label_df))
    if n_samples <= 0:
        continue
    sampled = label_df.sample(n=n_samples, random_state=42)
    sampled_dfs.append(sampled)

# Concatenate sampled subsets (handle the empty case)
if len(sampled_dfs) == 0:
    df_sampled = pd.DataFrame(columns=df_combined.columns)
else:
    df_sampled = pd.concat(sampled_dfs, ignore_index=True)

# Sanity checks
assert ' Label' in df_sampled.columns, "Expected ' Label' column in sampled dataframe"
print(f"Total sampled: {len(df_sampled)}")
print(df_sampled[' Label'].value_counts().to_string())

Total sampled: 14182
 Label
BENIGN                        2000
FTP - Patator                 2000
SSH - Patator                 2000
DDoS                          2000
PortScan                      2000
Bot                           1966
Web Attack - Brute Force      1507
Web Attack - XSS               652
Infiltration                    36
Web Attack - Sql Injection      21


In [8]:
columns_to_keep = [
    ' Label',
    ' Destination Port',
    ' Flow Duration',
    ' Total Fwd Packets',
    ' Total Backward Packets',
    ' SYN Flag Count',
    ' RST Flag Count',
    'FIN Flag Count',
    'Flow Bytes/s',
    ' Flow Packets/s',
    ' Flow IAT Mean',
    ' Down/Up Ratio',
    ' Average Packet Size',
    ' Packet Length Std',
    'Active Mean',
    'Idle Mean'
]

df_filtered = df_sampled[columns_to_keep]
print(f"Filtered dataframe shape: {df_filtered.shape}")
print(f"Columns: {df_filtered.columns.tolist()}")

Filtered dataframe shape: (14182, 16)
Columns: [' Label', ' Destination Port', ' Flow Duration', ' Total Fwd Packets', ' Total Backward Packets', ' SYN Flag Count', ' RST Flag Count', 'FIN Flag Count', 'Flow Bytes/s', ' Flow Packets/s', ' Flow IAT Mean', ' Down/Up Ratio', ' Average Packet Size', ' Packet Length Std', 'Active Mean', 'Idle Mean']


In [9]:
# Load the ATT&CK STIX bundle once and expose `attck_techniques` dict for reuse
with open('../data/attck/enterprise-attack.json', 'r', encoding='utf-8') as f:
    bundle = json.load(f)

# Dictionary to store techniques keyed by MITRE ID
attck_techniques = {}

# Iterate through all objects in the bundle
for obj in bundle.get('objects', []):
    if obj.get('type') != 'attack-pattern':
        continue
    if obj.get('revoked') or obj.get('x_mitre_deprecated'):
        continue
    tid = None
    for ref in obj.get('external_references', []):
        if ref.get('source_name') == 'mitre-attack':
            tid = ref.get('external_id')
            break
    if not tid:
        continue
    name = obj.get('name', 'Unknown')
    tactics = []
    for phase in obj.get('kill_chain_phases', []):
        phase_name = phase.get('phase_name', '')
        if phase_name:
            tactics.append(phase_name.capitalize())
    attck_techniques[tid] = {
        'name': name,
        'tactics': tactics,
        'description': obj.get('description','')
    }

# Display 3 sample techniques
sample_count = 0
for technique_id, technique_data in attck_techniques.items():
    if sample_count >= 3:
        break
    print(f"Technique ID: {technique_id}")
    print(f"  Name: {technique_data['name']}")
    print(f"  Tactics: {technique_data['tactics']}")
    print()
    sample_count += 1

Technique ID: T1055.011
  Name: Extra Window Memory Injection
  Tactics: ['Defense-evasion', 'Privilege-escalation']

Technique ID: T1053.005
  Name: Scheduled Task
  Tactics: ['Execution', 'Persistence', 'Privilege-escalation']

Technique ID: T1205.002
  Name: Socket Filters
  Tactics: ['Defense-evasion', 'Persistence', 'Command-and-control']



In [10]:
# Define the mapping from CICIDS labels to MITRE ATT&CK technique IDs
label_to_technique = {
    'BENIGN' : None,
    'FTP - Patator': 'T1110.001',      
    'SSH - Patator': 'T1110.001',       
    'DDoS': 'T1498.001',                
    'PortScan': 'T1046',                
    'Bot': 'T1071.001',                 
    'Web Attack - Brute Force': 'T1110.001', 
    'Web Attack - XSS': 'T1187',        
    'Infiltration': 'T1105',            
    'Web Attack - Sql Injection': 'T1190'  
}

In [11]:
# Map labels to MITRE technique IDs
df_sampled['Technique'] = df_sampled[' Label'].map(label_to_technique)

In [12]:
# retrieve tactics from technique ID using the single loaded ATT&CK dict
def get_tactics(tech_id):
    if pd.isna(tech_id):
        return None
    return attck_techniques.get(tech_id, {}).get('tactics', None)

df_sampled['Tactics'] = df_sampled['Technique'].apply(get_tactics)

In [13]:
# The 2000 benign samples, hence remaining data is mapped
print(df_sampled['Technique'].isna().sum())

2000


In [14]:
df_sampled.head()

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,Source_File,Technique,Tactics
0,53,808,2,2,102,256,51,51,51.00,0.000000,...,0,0,0.0,0.0,0,0,BENIGN,Friday-WorkingHours-Morning.pcap_ISCX.csv,NaN,None
1,53,826,2,2,58,178,29,29,29.00,0.000000,...,0,0,0.0,0.0,0,0,BENIGN,Thursday-WorkingHours-Afternoon-Infilteration....,NaN,None
2,7815,38,1,1,0,0,0,0,0.00,0.000000,...,0,0,0.0,0.0,0,0,BENIGN,Tuesday-WorkingHours.pcap_ISCX.csv,NaN,None
3,443,5395624,16,23,772,27243,455,0,48.25,121.691687,...,0,0,0.0,0.0,0,0,BENIGN,Tuesday-WorkingHours.pcap_ISCX.csv,NaN,None
4,80,5882540,3,1,12,0,6,0,4.00,3.464102,...,0,0,0.0,0.0,0,0,BENIGN,Thursday-WorkingHours-Afternoon-Infilteration....,NaN,None


In [15]:
def get_port_service(port: int) -> str:
    """Maps common ports to service names for semantic grounding."""
    port_map = {
    20: "FTP-Data",
    21: "FTP",
    22: "SSH",
    23: "Telnet",
    25: "SMTP",
    53: "DNS",
    67: "DHCP-Server",
    68: "DHCP-Client",
    69: "TFTP",
    80: "HTTP",
    110: "POP3",
    119: "NNTP",
    123: "NTP",
    135: "RPC",
    137: "NetBIOS-Name",
    138: "NetBIOS-Datagram",
    139: "NetBIOS-Session",
    143: "IMAP",
    161: "SNMP",
    162: "SNMP-Trap",
    179: "BGP",
    389: "LDAP",
    443: "HTTPS",
    445: "SMB",
    465: "SMTPS",
    514: "Syslog",
    587: "SMTP-Submission",
    636: "LDAPS",
    993: "IMAPS",
    995: "POP3S",
    1080: "SOCKS Proxy",
    1433: "MSSQL",
    1521: "Oracle DB",
    1723: "PPTP",
    2049: "NFS",
    3306: "MySQL",
    3389: "RDP",
    5432: "PostgreSQL",
    5900: "VNC",
    6379: "Redis",
    8080: "HTTP-Alt",
    8443: "HTTPS-Alt",
    9000: "SonarQube/Dev services",
    9200: "Elasticsearch",
    27017: "MongoDB"
}

    return port_map.get(port, f"unknown-port")

def describe_duration(dur: float) -> str:
    if dur < 1.0: return "brief, sub-second connection"
    elif dur < 10.0: return "short-lived connection"
    elif dur < 60.0: return "sustained interaction"
    elif dur < 300.0: return "persistent session"
    else: return "long-duration connection"

def describe_volume(fwd: int, bwd: int) -> str:
    total = fwd + bwd
    if total < 10: return "minimal control exchange"
    elif total < 100: return "light data transfer"
    elif total < 1000: return "moderate communication"
    else: return "high-volume data stream"

def describe_flags(syn: int, rst: int, fin: int) -> str:
    if syn > 50 and rst == 0 and fin == 0:
        return "repeated connection attempts without completion (scanning or brute-force behavior)"
    elif rst > syn * 0.4:
        return "frequent connection resets (potential evasion, policy blocking, or connection instability)"
    elif fin > 0 and rst == 0:
        return "graceful connection termination"
    else:
        return "mixed TCP control behavior with partial handshakes"

def describe_asymmetry(ratio: float) -> str:
    if ratio < 0.1:
        return "highly asymmetric outbound traffic (command flooding or data exfiltration pattern)"
    elif ratio > 10.0:
        return "highly asymmetric inbound traffic (bulk download or beacon response pattern)"
    else:
        return "balanced bidirectional communication"

def describe_iat(iat: float) -> str:
    if iat < 0.01: return "extremely rapid, machine-speed pacing"
    elif iat < 0.1: return "fast automated timing"
    elif iat < 1.0: return "steady interactive pacing"
    else: return "sporadic or human-like intervals"

def describe_session(act: float, idle: float) -> str:
    if act > 0 and idle < 1: return "continuous active session"
    elif idle > act * 2: return "intermittent beaconing with long dormant periods"
    else: return "regular active-idle communication cycle"

def describe_payload(avg: float, std: float) -> str:
    if avg < 100: return "small control or command packets"
    elif avg > 1000: return "large data payload transfers"
    else: return "moderate mixed-size packets"

In [16]:
def build_semantic_alert(row: pd.Series) -> str:
    port    = int(row[' Destination Port']) if pd.notna(row[' Destination Port']) else 0
    dur     = float(row[' Flow Duration']) if pd.notna(row[' Flow Duration']) else 0.0
    fwd     = int(row[' Total Fwd Packets']) if pd.notna(row[' Total Fwd Packets']) else 0
    bwd     = int(row[' Total Backward Packets']) if pd.notna(row[' Total Backward Packets']) else 0
    syn     = int(row[' SYN Flag Count']) if pd.notna(row[' SYN Flag Count']) else 0
    rst     = int(row[' RST Flag Count']) if pd.notna(row[' RST Flag Count']) else 0
    fin     = int(row['FIN Flag Count']) if pd.notna(row['FIN Flag Count']) else 0
    pps     = float(row[' Flow Packets/s']) if pd.notna(row[' Flow Packets/s']) else 0.0
    bps     = float(row['Flow Bytes/s']) if pd.notna(row['Flow Bytes/s']) else 0.0
    iat     = float(row[' Flow IAT Mean']) if pd.notna(row[' Flow IAT Mean']) else 0.0
    asym    = float(row[' Down/Up Ratio']) if pd.notna(row[' Down/Up Ratio']) else 0.0
    avg_sz  = float(row[' Average Packet Size']) if pd.notna(row[' Average Packet Size']) else 0.0
    std_sz  = float(row[' Packet Length Std']) if pd.notna(row[' Packet Length Std']) else 0.0
    act     = float(row['Active Mean']) if pd.notna(row['Active Mean']) else 0.0
    idle    = float(row['Idle Mean']) if pd.notna(row['Idle Mean']) else 0.0

    return (
        f"Network flow targeting {get_port_service(port)} on port {port}. "
        f"Connection profile: {describe_duration(dur)}. "
        f"Traffic exchange: {describe_volume(fwd, bwd)}, with {fwd} outbound and {bwd} inbound packets. "
        f"TCP behavior: {describe_flags(syn, rst, fin)}. "
        f"Timing pattern: {describe_iat(iat)}. "
        f"Directionality: {describe_asymmetry(asym)}. "
        f"Session rhythm: {describe_session(act, idle)}. "
        f"Payload characteristics: {describe_payload(avg_sz, std_sz)}."
    )

In [17]:
# Apply to your filtered DataFrame
df_filtered['alert_text'] = df_filtered.apply(build_semantic_alert, axis=1)
alert_texts = df_filtered['alert_text'].tolist()

In [18]:

pd.set_option('display.max_colwidth', None)
print(df_filtered['alert_text'].head())


0                                                        Network flow targeting DNS on port 53. Connection profile: long-duration connection. Traffic exchange: minimal control exchange, with 2 outbound and 2 inbound packets. TCP behavior: mixed TCP control behavior with partial handshakes. Timing pattern: sporadic or human-like intervals. Directionality: balanced bidirectional communication. Session rhythm: regular active-idle communication cycle. Payload characteristics: moderate mixed-size packets.
1                                                   Network flow targeting DNS on port 53. Connection profile: long-duration connection. Traffic exchange: minimal control exchange, with 2 outbound and 2 inbound packets. TCP behavior: mixed TCP control behavior with partial handshakes. Timing pattern: sporadic or human-like intervals. Directionality: balanced bidirectional communication. Session rhythm: regular active-idle communication cycle. Payload characteristics: small control or comma

In [19]:
from sentence_transformers import SentenceTransformer, util
embedder = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = embedder.encode(alert_texts, convert_to_tensor=True)
communities = util.community_detection(embeddings, min_community_size=2, threshold=0.75)
print(f"Number of communities: {len(communities)}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Number of communities: 15


In [20]:
import numpy as np

# build positional array and assign
comm_ids = np.full(len(df_filtered), -1, dtype=int)
for cid, members in enumerate(communities):
    comm_ids[np.asarray(members, dtype=int)] = cid
df_filtered['community_id'] = comm_ids

# validations
assigned_count = (df_filtered['community_id'] != -1).sum()
assert assigned_count == sum(len(c) for c in communities), "Assigned count mismatch"
flat = [i for c in communities for i in c]
assert len(set(flat)) == len(flat), "Overlapping indices in communities"
print("Assigned", assigned_count, "clustered rows; unclustered =", (len(df_filtered)-assigned_count))
print(df_filtered['community_id'].value_counts(dropna=False).head(10))

Assigned 14178 clustered rows; unclustered = 4
community_id
 0    13451
 1      312
 2      247
 3      122
 4       12
 5        7
 6        7
-1        4
 7        4
 9        3
Name: count, dtype: int64


In [21]:
from pathlib import Path
# Map Technique/Tactics into df_filtered directly and save — df_filtered is canonical
if 'df_sampled' not in globals():
    raise RuntimeError('df_sampled missing — run sampling cell first')
# Ensure df_sampled has Technique/Tactics columns (populate if absent)
if 'Technique' not in df_sampled.columns:
    if 'label_to_technique' in globals():
        df_sampled['Technique'] = df_sampled[' Label'].map(label_to_technique)
    else:
        df_sampled['Technique'] = None
if 'Tactics' not in df_sampled.columns:
    if 'attck_techniques' in globals():
        df_sampled['Tactics'] = df_sampled['Technique'].apply(lambda tid: attck_techniques.get(tid, {}).get('tactics') if pd.notna(tid) else None)
    else:
        df_sampled['Tactics'] = None
# Assign into df_filtered aligned by index — df_filtered becomes canonical processed table
df_filtered['Technique'] = df_sampled.loc[df_filtered.index, 'Technique'].values
df_filtered['Tactics']   = df_sampled.loc[df_filtered.index, 'Tactics'].apply(str).values
RESULTS_DIR = Path('../data/results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
df_filtered.to_csv(RESULTS_DIR / 'cicids_processed.csv', index=False)
print(f'Saved {len(df_filtered)} rows → cicids_processed.csv')

Saved 14182 rows → cicids_processed.csv


In [22]:
# Stage 2 — Triplet Extraction → Knowledge Graph → RAG Reports
# (Top-level imports handled in first cell; this cell coordinates stage-2 variables and checks)
RESULTS_DIR = Path('../data/results')
ATTCK_DIR   = Path('../data/attck')
CHROMA_DIR  = Path('../data/chroma')
for d in [RESULTS_DIR, CHROMA_DIR]:
    d.mkdir(parents=True, exist_ok=True)
# community_df will be created later from clustered rows

In [23]:
# LLM initialization (uses Llama imported at top)
llm = Llama(
    model_path=MODEL_PATH,
    n_ctx=4096, n_gpu_layers=-1, n_threads=8,
    n_batch=256, verbose=False, seed=RANDOM_SEED,
    )
print(f'LLM loaded: {MODEL_PATH}')

llama_context: n_ctx_seq (4096) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


LLM loaded: ../models/qwen2.5-3b-instruct-q4_k_m.gguf


In [24]:
SECURITY_RELATIONS = [
    'PERFORMS_RECONNAISSANCE',
    'PERFORMS_PORT_SCAN',
    'BRUTE_FORCES_CREDENTIAL',
    'ACCESS_CREDENTIALS',
    'EXPLOITS_VULNERABILITY',
    'ESTABLISHES_C2',
    'PERFORMS_BEACONING',
    'CAUSES_DENIAL_OF_SERVICE',
    'MOVES_LATERALLY',
    'EXFILTRATES_DATA',
    'EXECUTES_PAYLOAD',
]

TRIPLE_SCHEMA = {
    'type': 'object',
    'properties': {
        'triples': {
            'type': 'array', 'minItems': 4, 'maxItems': 4,
            'items': {
                'type': 'object',
                'properties': {
                    'subject':  {'type': 'string'},
                    'relation': {'type': 'string', 'enum': SECURITY_RELATIONS},
                    'target':   {'type': 'string'}
                },
                'required': ['subject', 'relation', 'target']
            }
        }
    },
    'required': ['triples']
}

grammar = LlamaGrammar.from_json_schema(json.dumps(TRIPLE_SCHEMA))
print(f'Grammar ready — {len(SECURITY_RELATIONS)} constrained relations')

Grammar ready — 11 constrained relations


In [25]:
RELATION_GUIDE = """
Relation definitions (use exactly as written):
  PERFORMS_RECONNAISSANCE  : active discovery, general service enumeration and scanning
  PERFORMS_PORT_SCAN       : focused port/protocol probing across multiple ports or hosts
  BRUTE_FORCES_CREDENTIAL  : repeated authentication attempts against a service (password guessing)
  ACCESS_CREDENTIALS       : theft or collection of credentials (beyond brute-force, e.g., credential dumping or theft)
  EXPLOITS_VULNERABILITY   : exploitation of a software or configuration flaw
  ESTABLISHES_C2           : outbound communication to a command-and-control channel
  PERFORMS_BEACONING       : periodic callback/beacon patterns indicative of persistent C2
  CAUSES_DENIAL_OF_SERVICE : flooding or resource exhaustion of a target service
  MOVES_LATERALLY          : accessing internal hosts after initial compromise
  EXFILTRATES_DATA         : transferring data out of the environment
  EXECUTES_PAYLOAD         : running code or commands on a target
""".strip()


def normalise_entity(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r'[^a-z0-9_]+', '_', text)
    text = re.sub(r'_+', '_', text).strip('_')
    return text[:80]


In [26]:
def extract_triples(alert_texts: list, community_id: int) -> list:
    """Extract exactly 4 validated triples from a list of alert_texts.
    Uses the LLM grammar to constrain relation values. Returns a list of 4 dicts
    with keys: subject, relation, target. If the LLM response cannot be parsed,
    returns fallback triples to preserve downstream pipeline guarantees.
    """
    if 'llm' not in globals():
        raise RuntimeError('LLM not initialized. Run the cell that creates `llm` before extracting triples.')
    block = '\n'.join(f'- {t}' for t in alert_texts[:6])

    # Guard: ensure no dataset label/technique fields are present in the prompt block
    assert ' Label' not in block and 'Technique' not in block, \
        f"Potential label leakage detected in extract_triples block for community {community_id}"

    prompt = f"""[INST] You are a cybersecurity analyst performing threat analysis.
Read the network alerts below and extract exactly 4 semantic triples describing
the attack behaviour using standard security terminology.

{RELATION_GUIDE}

For each triple:
- subject : name the specific actor observed (e.g. 'ssh_scanner', 'ddos_flood_source')
- relation: choose the ONE relation that best fits — use payload size, flag patterns,
            packet rate, IAT timing, and session rhythm to decide
- target  : name the specific service or resource (e.g. 'ssh_port_22', 'http_web_server')

Name what you observe. No generic placeholders like 'source' or 'destination'.

Example (SSH brute-force):
{{"triples": [
  {{"subject": "ssh_brute_force_client",      "relation": "BRUTE_FORCES_CREDENTIAL", "target": "ssh_port_22_service"}},
  {{"subject": "ssh_brute_force_client",      "relation": "PERFORMS_RECONNAISSANCE", "target": "ssh_authentication_endpoint"}},
  {{"subject": "credential_guessing_process", "relation": "EXECUTES_PAYLOAD",        "target": "password_spray_module"}},
  {{"subject": "attacker",                    "relation": "BRUTE_FORCES_CREDENTIAL", "target": "user_account_store"}}
]}}

ALERTS (Community {community_id}):
{block}

Return ONLY valid JSON. [/INST]"""
    out = llm(prompt, max_tokens=512, temperature=0, seed=RANDOM_SEED,
              grammar=grammar, repeat_penalty=1.1, stop=['[/INST]'])
    raw = out['choices'][0]['text'].strip()

    try:
        triples = json.loads(raw).get('triples', [])
    except Exception as e:
        print(f'  [Community {community_id}] parse failed: {e}')
        triples = []

    valid_relations = set(SECURITY_RELATIONS)
    validated = []
    for t in triples:
        subj = normalise_entity(str(t.get('subject', '')))
        rel  = str(t.get('relation', '')).upper().strip()
        tgt  = normalise_entity(str(t.get('target', '')))
        if not subj or not tgt:
            continue
        if rel not in valid_relations:
            rel = 'PERFORMS_RECONNAISSANCE'
        validated.append({'subject': subj, 'relation': rel, 'target': tgt})

    # Create unique fallback triples when extraction fails/insufficient
    while len(validated) < 4:
        idx = len(validated)
        validated.append({
            'subject': f'community_{community_id}_actor_{idx}',
            'relation': 'PERFORMS_RECONNAISSANCE',
            'target': f'community_{community_id}_target_{idx}'
        })
    return validated[:4]

In [27]:
# Build `community_df` as a transient view of `df_filtered` (df_filtered is canonical)
if 'df_filtered' not in globals():
    raise RuntimeError('Missing df_filtered — run preprocessing cells first')
if 'community_id' not in df_filtered.columns:
    raise RuntimeError('`community_id` not found in df_filtered — run the community detection cell')
# community_df is a filtered copy of clustered rows only
community_df = df_filtered[df_filtered['community_id'] != -1].reset_index(drop=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
community_df.to_csv(RESULTS_DIR / 'community_assignments.csv', index=False)
print(f'Prepared community_df with {len(community_df)} clustered alerts across {community_df["community_id"].nunique()} communities')

print('Warming up LLM...')
_ = llm('[INST] Test. [/INST]', max_tokens=5, temperature=0, seed=RANDOM_SEED)
print('Starting extraction\n')

community_triples = {}

for cid, group in community_df.groupby('community_id'):
    texts   = group['alert_text'].dropna().tolist()
    triples = extract_triples(texts, int(cid))
    community_triples[str(int(cid))] = triples

    dom_label = group[' Label'].mode().iloc[0]  # informational only — never in prompt
    print(f'Community {cid} [{dom_label}]:')
    for t in triples:
        print(f"  {t['subject']} --[{t['relation']}]-- {t['target']}")
    print()

print(f'Extraction complete: {len(community_triples)} communities')

Prepared community_df with 14178 clustered alerts across 15 communities
Warming up LLM...
Starting extraction

Community 0 [DDoS]:
  dns_brute_force_attacker --[BRUTE_FORCES_CREDENTIAL]-- dns_port_53_service
  dns_brute_force_attacker --[PERFORMS_RECONNAISSANCE]-- dns_authentication_endpoint
  command_flooding_botnet --[EXFILTRATES_DATA]-- http_web_server
  unknown_port_exploit_actor --[EXPLOITS_VULNERABILITY]-- unknown_port_service

Community 1 [BENIGN]:
  https_exfiltration_actor --[EXFILTRATES_DATA]-- https_web_server
  https_exfiltration_actor --[PERFORMS_BEACONING]-- https_web_server
  https_exfiltration_actor --[CAUSES_DENIAL_OF_SERVICE]-- https_web_server
  https_exfiltration_actor --[EXECUTES_PAYLOAD]-- command_and_control_module

Community 2 [BENIGN]:
  dns_probe --[PERFORMS_RECONNAISSANCE]-- dns_server
  dns_probe --[ESTABLISHES_C2]-- unknown_c2_channel
  dns_probe --[MOVES_LATERALLY]-- internal_network
  dns_probe --[EXFILTRATES_DATA]-- user_data

Community 3 [BENIGN]:
  htt

In [28]:
all_subjects = set()
all_targets  = set()
rel_counts   = Counter()
valid_count  = total_count = 0

for triples in community_triples.values():
    for t in triples:
        total_count += 1
        if all(k in t and t[k] for k in ['subject', 'relation', 'target']):
            valid_count += 1
            all_subjects.add(t['subject'])
            all_targets.add(t['target'])
            rel_counts[t['relation']] += 1

triple_metrics = {
    'n_communities':         len(community_triples),
    'total_triples':         total_count,
    'valid_triples':         valid_count,
    'valid_ratio':           round(valid_count / max(total_count, 1), 4),
    'unique_subjects':       len(all_subjects),
    'unique_targets':        len(all_targets),
    'unique_entities_total': len(all_subjects | all_targets),
    'relation_counts':       dict(rel_counts),
    'relation_coverage':     f'{len(rel_counts)}/{len(SECURITY_RELATIONS)} ontology relations used',
}
with open(RESULTS_DIR / 'community_triples.json', 'w', encoding='utf-8') as f:
    json.dump(community_triples, f, indent=2)
with open(RESULTS_DIR / 'triple_metrics.json', 'w') as f:
    json.dump(triple_metrics, f, indent=2)

print('TRIPLE METRICS')
for k, v in triple_metrics.items():
    print(f'  {k}: {v}')

TRIPLE METRICS
  n_communities: 15
  total_triples: 60
  valid_triples: 60
  valid_ratio: 1.0
  unique_subjects: 20
  unique_targets: 34
  unique_entities_total: 53
  relation_counts: {'BRUTE_FORCES_CREDENTIAL': 7, 'PERFORMS_RECONNAISSANCE': 17, 'EXFILTRATES_DATA': 10, 'EXPLOITS_VULNERABILITY': 2, 'PERFORMS_BEACONING': 3, 'CAUSES_DENIAL_OF_SERVICE': 3, 'EXECUTES_PAYLOAD': 6, 'ESTABLISHES_C2': 4, 'MOVES_LATERALLY': 2, 'PERFORMS_PORT_SCAN': 2, 'ACCESS_CREDENTIALS': 4}
  relation_coverage: 11/11 ontology relations used


In [29]:
# Build graph reusing previously-loaded ATT&CK techniques and add bridge edges idempotently
G = nx.DiGraph()

# Behavioral layer
edge_weights     = {}
edge_communities = {}
total_triples    = 0
invalid_triples  = 0

for cid, triples in community_triples.items():
    for t in triples:
        s = str(t.get('subject',  '')).strip()
        r = str(t.get('relation', '')).strip()
        o = str(t.get('target',   '')).strip()
        if not s or not r or not o:
            invalid_triples += 1
            continue
        total_triples += 1
        key = (s, r, o)
        edge_weights[key] = edge_weights.get(key, 0) + 1
        edge_communities.setdefault(key, []).append(cid)

for (src, rel, tgt), weight in edge_weights.items():
    G.add_node(src, layer='behavioral')
    G.add_node(tgt, layer='behavioral')
    G.add_edge(src, tgt, relation=rel, weight=weight, layer='behavioral',
               communities=','.join(edge_communities[(src, rel, tgt)]))

# ATT&CK layer — rely on the single `attck_techniques` loaded earlier
if 'attck_techniques' not in globals():
    raise RuntimeError('ATT&CK techniques not loaded — run the ATT&CK load cell first.')
else:
    print(f'Reusing existing `attck_techniques` ({len(attck_techniques)})')

# Add ATT&CK nodes idempotently
for tid, info in attck_techniques.items():
    if tid not in G:
        G.add_node(tid, layer='attck', name=info.get('name',''),
                   tactic=(info.get('tactics') or [''])[0], description=info.get('description',''))

# Ensure RELATION_TO_TECHNIQUES mapping exists (fallback to default mapping)
if 'RELATION_TO_TECHNIQUES' not in globals():
    RELATION_TO_TECHNIQUES = {
        'PERFORMS_RECONNAISSANCE':  ['T1046',    'T1595',    'T1590'],
        'PERFORMS_PORT_SCAN':       ['T1046',    'T1595'],
        'BRUTE_FORCES_CREDENTIAL':  ['T1110',    'T1110.001','T1110.003'],
        'ACCESS_CREDENTIALS':       ['T1555',    'T1078',    'T1110'],
        'EXPLOITS_VULNERABILITY':   ['T1190',    'T1203'],
        'ESTABLISHES_C2':           ['T1071',    'T1071.001','T1071.004'],
        'PERFORMS_BEACONING':       ['T1071',    'T1071.004'],
        'CAUSES_DENIAL_OF_SERVICE': ['T1498',    'T1499',    'T1499.001'],
        'MOVES_LATERALLY':          ['T1021',    'T1570'],
        'EXFILTRATES_DATA':         ['T1041',    'T1048'],
        'EXECUTES_PAYLOAD':         ['T1059',    'T1059.007','T1105'],
    }
    print('Inserted default RELATION_TO_TECHNIQUES mapping')

# Bridge layer — connect behavioural nodes to ATT&CK technique nodes using RELATION_TO_TECHNIQUES mapping
bridge_count = 0
for (src, rel, tgt), weight in edge_weights.items():
    for tid in RELATION_TO_TECHNIQUES.get(rel, []):
        if tid in G and not G.has_edge(src, tid):
            G.add_edge(src, tid, relation='ASSOCIATED_WITH', weight=weight,
                       layer='bridge', communities=','.join(edge_communities.get((src, rel, tgt), [])))
            bridge_count += 1

b_nodes = sum(1 for n, d in G.nodes(data=True) if d.get('layer') == 'behavioral')
b_edges = sum(1 for _, _, d in G.edges(data=True) if d.get('layer') == 'behavioral')
print(f'Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
print(f'  Behavioral : {b_nodes} nodes, {b_edges} edges')
print(f'  ATT&CK     : {len(attck_techniques)} technique nodes')
print(f'  Bridge     : {bridge_count} edges | Skipped: {invalid_triples} invalid triples')

Reusing existing `attck_techniques` (691)
Inserted default RELATION_TO_TECHNIQUES mapping
Graph: 744 nodes, 195 edges
  Behavioral : 53 nodes, 51 edges
  ATT&CK     : 691 technique nodes
  Bridge     : 144 edges | Skipped: 0 invalid triples


In [30]:
beh_nodes = [(n, d) for n, d in G.degree() if G.nodes[n].get('layer') == 'behavioral']
print('TOP 10 BEHAVIORAL NODES BY DEGREE')
for node, deg in sorted(beh_nodes, key=lambda x: x[1], reverse=True)[:10]:
    print(f'  {node} (degree={deg})')

print('\nTOP 10 BEHAVIORAL EDGES BY WEIGHT')
beh_edges = [(u, v, d['relation'], d['weight']) for u, v, d in G.edges(data=True)
             if d.get('layer') == 'behavioral']
for u, v, rel, w in sorted(beh_edges, key=lambda x: x[3], reverse=True)[:10]:
    print(f'  (w={w}) {u} --[{rel}]--> {v}')

reachable_attck = [n for n in G.nodes()
                   if G.nodes[n].get('layer') == 'attck' and G.in_degree(n) > 0]
print(f'\nATT&CK nodes reachable: {len(reachable_attck)}')
for tid in sorted(reachable_attck):
    print(f"  {tid} | {G.nodes[tid].get('name')} | {G.nodes[tid].get('tactic')}")

TOP 10 BEHAVIORAL NODES BY DEGREE
  https_traffic_source (degree=19)
  ldap_brute_force_attacker (degree=16)
  rdp_brute_force_attacker (degree=15)
  dns_probe (degree=14)
  imap_brute_force_attacker (degree=13)
  https_exfiltration_actor (degree=12)
  smtp_submission_client (degree=12)
  https_probe_client (degree=11)
  attacker (degree=11)
  sonarqube_dev_services (degree=10)

TOP 10 BEHAVIORAL EDGES BY WEIGHT
  (w=2) https_traffic_source --[PERFORMS_RECONNAISSANCE]--> https_web_server
  (w=2) attacker --[PERFORMS_RECONNAISSANCE]--> https_web_server
  (w=1) dns_brute_force_attacker --[BRUTE_FORCES_CREDENTIAL]--> dns_port_53_service
  (w=1) dns_brute_force_attacker --[PERFORMS_RECONNAISSANCE]--> dns_authentication_endpoint
  (w=1) command_flooding_botnet --[EXFILTRATES_DATA]--> http_web_server
  (w=1) command_flooding_botnet --[CAUSES_DENIAL_OF_SERVICE]--> ldap_port_389_service
  (w=1) unknown_port_exploit_actor --[EXPLOITS_VULNERABILITY]--> unknown_port_service
  (w=1) https_exfiltra

In [31]:
nx.write_graphml(G, RESULTS_DIR / 'knowledge_graph.graphml')

nodes_out = [{'node': n, 'degree': G.degree(n), 'layer': d.get('layer', ''),
              'name': d.get('name', ''), 'tactic': d.get('tactic', ''),
              'description': d.get('description', '')}
             for n, d in G.nodes(data=True)]
pd.DataFrame(nodes_out).sort_values('degree', ascending=False).to_csv(
    RESULTS_DIR / 'knowledge_graph_nodes.csv', index=False)

edges_out = [{'source': u, 'target': v, 'relation': d['relation'],
              'weight': d['weight'], 'layer': d.get('layer', ''),
              'communities': d.get('communities', '')}
             for u, v, d in G.edges(data=True)]
kg_edges_df = pd.DataFrame(edges_out).sort_values('weight', ascending=False)
kg_edges_df.to_csv(RESULTS_DIR / 'knowledge_graph_edges.csv', index=False)

rel_dist = Counter(d['relation'] for _, _, d in G.edges(data=True))
kg_metrics = {
    'total_triples_processed': total_triples,
    'invalid_triples_skipped': invalid_triples,
    'behavioral_nodes':        b_nodes,
    'attck_technique_nodes':   len(attck_techniques),
    'total_nodes':             G.number_of_nodes(),
    'behavioral_edges':        b_edges,
    'bridge_edges':            bridge_count,
    'total_edges':             G.number_of_edges(),
    'attck_nodes_reachable':   len(reachable_attck),
    'relation_distribution':   dict(rel_dist),
}
with open(RESULTS_DIR / 'knowledge_graph_metrics.json', 'w') as f:
    json.dump(kg_metrics, f, indent=2)

print('Graph saved. Metrics:')
for k, v in kg_metrics.items():
    print(f'  {k}: {v}')

Graph saved. Metrics:
  total_triples_processed: 60
  invalid_triples_skipped: 0
  behavioral_nodes: 53
  attck_technique_nodes: 691
  total_nodes: 744
  behavioral_edges: 51
  bridge_edges: 144
  total_edges: 195
  attck_nodes_reachable: 23
  relation_distribution: {'BRUTE_FORCES_CREDENTIAL': 5, 'PERFORMS_RECONNAISSANCE': 14, 'ASSOCIATED_WITH': 144, 'EXFILTRATES_DATA': 9, 'CAUSES_DENIAL_OF_SERVICE': 3, 'EXPLOITS_VULNERABILITY': 2, 'EXECUTES_PAYLOAD': 6, 'ESTABLISHES_C2': 4, 'MOVES_LATERALLY': 2, 'PERFORMS_PORT_SCAN': 1, 'PERFORMS_BEACONING': 1, 'ACCESS_CREDENTIALS': 4}


In [32]:
EMBED_MODEL       = 'all-MiniLM-L6-v2'
TOP_K             = 10
MAX_CHARS_PER_DOC = 400

ef         = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=EMBED_MODEL)
client     = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = client.get_or_create_collection(name='attack_techniques', embedding_function=ef)

if collection.count() == 0:
    print('Populating ChromaDB...')
    docs, ids, metas = [], [], []
    for tid, info in attck_techniques.items():
        # `attck_techniques` stores tactics as a list under 'tactics' — normalise to a single string
        tactic = (info.get('tactics') or [''])[0] if isinstance(info.get('tactics'), list) else (info.get('tactics') or '')
        text = (f"ID: {tid}\nName: {info.get('name','')}\n"
                f"Tactic: {tactic}\nDescription: {info.get('description','')}")
        docs.append(text)
        ids.append(tid)
        metas.append({'technique_id': tid, 'name': info.get('name',''), 'tactic': tactic})
    collection.add(ids=ids, documents=docs, metadatas=metas)
    print(f'Inserted {collection.count()} techniques')
else:
    print(f'ChromaDB ready: {collection.count()} technique descriptions')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


ChromaDB ready: 703 technique descriptions


In [33]:
REPORT_SCHEMA = {
    'type': 'object',
    'properties': {
        'technique_id': {'type': 'string'},
        'tactic':       {'type': 'string'},
        'summary':      {'type': 'string'},
        'evidence':     {'type': 'string'},
        'next_step':    {'type': 'string'}
    },
    'required': ['technique_id', 'tactic', 'summary', 'evidence', 'next_step']
}
report_grammar = LlamaGrammar.from_json_schema(json.dumps(REPORT_SCHEMA))
print('Report grammar ready.')

Report grammar ready.


In [34]:
def build_hyde_query(cid: int):
    """Returns (hyde_text, raw_triple_text, alert_context). No labels passed anywhere."""
    group   = community_df[community_df['community_id'] == cid]
    triples = community_triples.get(str(cid), [])

    triple_sentences = []
    for t in triples:
        s = t.get('subject',  '').replace('_', ' ')
        r = t.get('relation', '').replace('_', ' ').lower()
        o = t.get('target',   '').replace('_', ' ')
        if s and r and o:
            triple_sentences.append(f'{s} {r} {o}')
    raw_triple_text = '. '.join(triple_sentences)

    community_entities = ({t.get('subject', '') for t in triples} |
                          {t.get('target',  '') for t in triples})
    related = kg_edges_df[
        (kg_edges_df['source'].isin(community_entities) |
         kg_edges_df['target'].isin(community_entities)) &
        (kg_edges_df['layer'] == 'behavioral')
    ].head(3)
    graph_ctx = '; '.join(
        f"{r.source.replace('_',' ')} {r.relation.replace('_',' ').lower()} {r.target.replace('_',' ')}"
        for _, r in related.iterrows()
    ) or 'None identified'

    hyde_prompt = f"""[INST] You are a cybersecurity threat analyst.
Write a concise technical description of the attack technique based on the observed
network behaviours below. Write in MITRE ATT&CK style: what the adversary does,
which resources they target, observable indicators. Do NOT name a specific technique ID.

OBSERVED BEHAVIOURS:
{raw_triple_text}

RECURRING GRAPH PATTERNS:
{graph_ctx}

Write 3 to 5 sentences. [/INST]"""

    out = llm(hyde_prompt, max_tokens=200, temperature=0.2,
              seed=RANDOM_SEED, repeat_penalty=1.1, stop=['[/INST]'])
    hyde_query = out['choices'][0]['text'].strip()
    alert_context = ' '.join(group['alert_text'].dropna().head(3).tolist())

    # Guard: ensure no dataset label/technique fields are present in hyde query inputs
    assert ' Label' not in raw_triple_text and 'Technique' not in raw_triple_text, \
        f"Potential label leakage detected in raw_triple_text for community {cid}"
    assert ' Label' not in alert_context and 'Technique' not in alert_context, \
        f"Potential label leakage detected in alert_context for community {cid}"

    return hyde_query, raw_triple_text, alert_context

In [35]:
def generate_rag_report(hyde_query, raw_triples, alert_context, docs, metas):
    # Simple RAG report assembly using top retrieved technique as hypothesis
    top_id = 'Unknown'
    tactic = ''
    try:
        if metas and isinstance(metas, list):
            # prefer first metadata entry that contains a valid technique_id
            found = next((m for m in metas if isinstance(m, dict) and m.get('technique_id')), None)
            if found:
                top_meta = found
                top_id = top_meta.get('technique_id', 'Unknown')
                tactic = top_meta.get('tactic', '')
    except Exception:
        top_id = 'Unknown'; tactic = ''
    summary = (hyde_query or '').strip()[:400]
    evidence = '\n'.join(docs[:3]) if docs else ''
    next_step = 'Validate retrieval with telemetry and investigate correlated hosts.'
    return {'technique_id': top_id, 'tactic': tactic, 'summary': summary, 'evidence': evidence, 'next_step': next_step}


def generate_baseline_report(raw_triples):
    # Heuristic baseline: map relation keywords to candidate techniques using RELATION_TO_TECHNIQUES
    candidates = []
    try:
        text = raw_triples if isinstance(raw_triples, str) else ' '.join(raw_triples)
    except Exception:
        text = ''
    rel_map = globals().get('RELATION_TO_TECHNIQUES', {})
    for rel, tids in rel_map.items():
        if rel.replace('_',' ').lower() in text.lower():
            candidates.extend(tids)
    if not candidates:
        return {'technique_id':'Unknown','tactic':'','summary':'No baseline candidate','evidence':''}
    from collections import Counter
    top = Counter(candidates).most_common(1)[0][0]
    tactic = ''
    if 'attck_techniques' in globals():
        tactic = (attck_techniques.get(top,{}).get('tactics') or [''])[0]
    return {'technique_id': top, 'tactic': tactic, 'summary': 'Baseline heuristic selected '+top, 'evidence': ''}

In [36]:
def parent_match(gt_id: str, gen_id: str) -> bool:
    if not gt_id or not gen_id or gen_id == 'Unknown':
        return False
    return gt_id.split('.')[0] == gen_id.split('.')[0]


eval_df = community_df[
    community_df['Technique'].notna() &
    ~community_df['Technique'].astype(str).isin(['nan', 'None'])
]
eval_cids = sorted(eval_df['community_id'].unique())
print(f'Evaluating {len(eval_cids)} attack communities\n')

results = []

# Ensure technique embeddings available for local reranking
import numpy as np
from sentence_transformers import util as sutil

if 'tech_ids' not in globals() or 'tech_embs' not in globals():
    tech_ids = []
    tech_texts = []
    for tid, info in attck_techniques.items():
        tactic = (info.get('tactics') or [''])[0] if isinstance(info.get('tactics'), list) else (info.get('tactics') or '')
        text = f"ID: {tid}\nName: {info.get('name','')}\nTactic: {tactic}\nDescription: {info.get('description','')}"
        tech_ids.append(tid)
        tech_texts.append(text)
    tech_embs = embedder.encode(tech_texts, convert_to_tensor=True)

for cid in eval_cids:
    group        = community_df[community_df['community_id'] == cid]
    ground_truth = group['Technique'].mode().iloc[0]  # used ONLY for scoring

    hyde_query, raw_triples, alert_context = build_hyde_query(cid)

    # Hybrid retrieval: query with HyDE summary and raw triple text
    try:
        retrieval = collection.query(query_texts=[hyde_query, raw_triples], n_results=TOP_K)
        # collection returns lists per query; combine metas/docs from both
        docs_lists = retrieval.get('documents', [])
        metas_lists = retrieval.get('metadatas', [])
        docs = []
        metas = []
        for dl in docs_lists:
            docs.extend(dl or [])
        for ml in metas_lists:
            metas.extend(ml or [])
        # deduplicate metas by technique_id preserving order
        seen = set(); unique_metas = []
        for m in metas:
            tid = m.get('technique_id') if isinstance(m, dict) else None
            if tid and tid not in seen:
                unique_metas.append(m); seen.add(tid)
        metas = unique_metas
        retrieved_ids = [m.get('technique_id', '') for m in metas]
    except Exception as e:
        print('Collection query failed:', e)
        docs, metas, retrieved_ids = [], [], []

    # Local similarity reranking using HyDE and raw triples embeddings against technique embeddings
    try:
        q_hyde = embedder.encode(hyde_query or '', convert_to_tensor=True)
        q_raw  = embedder.encode(raw_triples or '', convert_to_tensor=True)
        sims_h = sutil.pytorch_cos_sim(q_hyde, tech_embs)[0].cpu().tolist()
        sims_r = sutil.pytorch_cos_sim(q_raw,  tech_embs)[0].cpu().tolist()
        combined = [max(h, r) for h, r in zip(sims_h, sims_r)]
        best_idx = int(np.argmax(combined))
        best_id = tech_ids[best_idx]
        best_score = float(combined[best_idx])
    except Exception as e:
        best_id = 'Unknown'; best_score = 0.0

    # Build RAG report using best_id and evidence from retrieved docs/metas
    rag_id = best_id
    rag_tactic = ''
    if rag_id != 'Unknown' and rag_id in attck_techniques:
        rag_tactic = (attck_techniques.get(rag_id, {}).get('tactics') or [''])[0]
    rag_report = {'technique_id': rag_id, 'tactic': rag_tactic, 'summary': (hyde_query or '')[:400], 'evidence': '\n'.join(docs[:3]) if docs else '', 'next_step': 'Validate retrieval with telemetry and investigate correlated hosts.'}

    base_report = generate_baseline_report(raw_triples)

    base_id = base_report.get('technique_id', 'Unknown')

    retrieval_hit = ground_truth in retrieved_ids
    # Consider 'Unknown' as NOT grounded; require a non-Unknown id present in retrievals
    rag_grounded  = (rag_id != 'Unknown') and (rag_id in retrieved_ids)
    rag_exact     = (rag_id == ground_truth) and rag_grounded
    rag_parent    = parent_match(ground_truth, rag_id) and rag_grounded
    base_exact    = base_id == ground_truth
    base_parent   = parent_match(ground_truth, base_id)

    results.append({
        'community_id':      cid,
        'ground_truth':      ground_truth,
        'dominant_label':    group[' Label'].mode().iloc[0],
        'dominant_tactic':   group['Tactics'].mode().iloc[0],
        'hyde_query':        hyde_query,
        'raw_triples':       raw_triples,
        'retrieved_ids':     retrieved_ids,
        'retrieval_hit':     retrieval_hit,
        'rag_technique_id':  rag_id,
        'rag_grounded':      rag_grounded,
        'rag_exact_match':   rag_exact,
        'rag_parent_match':  rag_parent,
        'rag_tactic':        rag_report.get('tactic',    ''),
        'rag_summary':       rag_report.get('summary',   ''),
        'rag_evidence':      rag_report.get('evidence',  ''),
        'rag_next_step':     rag_report.get('next_step', ''),
        'rag_score':         best_score,
        'base_technique_id': base_id,
        'base_exact_match':  base_exact,
        'base_parent_match': base_parent,
        'base_tactic':       base_report.get('tactic',  ''),
        'base_summary':      base_report.get('summary', ''),
    })

    print(f"  Community {cid} [{ground_truth}] | ")
    print(f"    RAG(best): {rag_id} (score={best_score:.3f}) | grounded={rag_grounded} | exact={rag_exact} | parent={rag_parent}")
    print(f"    Baseline:   {base_id} | exact={base_exact} | parent={base_parent}")

print(f'\nDone: {len(results)} communities evaluated')


Evaluating 11 attack communities

  Community 0 [T1110.001] | 
    RAG(best): T1071.004 (score=0.641) | grounded=True | exact=False | parent=False
    Baseline:   T1046 | exact=False | parent=False
  Community 1 [T1110.001] | 
    RAG(best): T1071.001 (score=0.720) | grounded=True | exact=False | parent=False
    Baseline:   T1071 | exact=False | parent=False
  Community 5 [T1046] | 
    RAG(best): T1205.001 (score=0.561) | grounded=True | exact=False | parent=False
    Baseline:   T1046 | exact=True | parent=True
  Community 6 [T1046] | 
    RAG(best): T1071.003 (score=0.494) | grounded=True | exact=False | parent=False
    Baseline:   T1046 | exact=True | parent=True
  Community 7 [T1046] | 
    RAG(best): T1563.002 (score=0.703) | grounded=True | exact=False | parent=False
    Baseline:   T1046 | exact=True | parent=True
  Community 8 [T1046] | 
    RAG(best): T1110.003 (score=0.556) | grounded=True | exact=False | parent=False
    Baseline:   T1046 | exact=True | parent=True
  Comm

In [37]:
n = len(results)

metrics = {
    'model':                      'qwen2.5-3b-instruct-q4_k_m',
    'retrieval_strategy':         'HyDE',
    'embed_model':                EMBED_MODEL,
    'n_communities_evaluated':    n,
    'top_k_retrieval':            TOP_K,
    'retrieval_hit_rate':         round(sum(r['retrieval_hit']    for r in results) / n, 4),
    'rag_grounding_rate':         round(sum(r['rag_grounded']     for r in results) / n, 4),
    'rag_exact_match_rate':       round(sum(r['rag_exact_match']  for r in results) / n, 4),
    'rag_parent_match_rate':      round(sum(r['rag_parent_match'] for r in results) / n, 4),
    'baseline_exact_match_rate':  round(sum(r['base_exact_match'] for r in results) / n, 4),
    'baseline_parent_match_rate': round(sum(r['base_parent_match']for r in results) / n, 4),
}
metrics['delta_exact_match']  = round(metrics['rag_exact_match_rate']  - metrics['baseline_exact_match_rate'],  4)
metrics['delta_parent_match'] = round(metrics['rag_parent_match_rate'] - metrics['baseline_parent_match_rate'], 4)

pd.DataFrame(results).to_csv(RESULTS_DIR / 'rag_reports.csv', index=False)
with open(RESULTS_DIR / 'rag_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print('METRICS')
print(json.dumps(metrics, indent=2))
print(f"\nRAG exact:       {metrics['rag_exact_match_rate']:.2%}")
print(f"Baseline exact:  {metrics['baseline_exact_match_rate']:.2%}")
print(f"Delta (exact):   {metrics['delta_exact_match']:+.2%}")
print(f"\nRAG parent:      {metrics['rag_parent_match_rate']:.2%}")
print(f"Baseline parent: {metrics['baseline_parent_match_rate']:.2%}")
print(f"Delta (parent):  {metrics['delta_parent_match']:+.2%}")
print(f"\nRetrieval hit:   {metrics['retrieval_hit_rate']:.2%}")

METRICS
{
  "model": "qwen2.5-3b-instruct-q4_k_m",
  "retrieval_strategy": "HyDE",
  "embed_model": "all-MiniLM-L6-v2",
  "n_communities_evaluated": 11,
  "top_k_retrieval": 10,
  "retrieval_hit_rate": 0.0,
  "rag_grounding_rate": 1.0,
  "rag_exact_match_rate": 0.0,
  "rag_parent_match_rate": 0.0,
  "baseline_exact_match_rate": 0.8182,
  "baseline_parent_match_rate": 0.8182,
  "delta_exact_match": -0.8182,
  "delta_parent_match": -0.8182
}

RAG exact:       0.00%
Baseline exact:  81.82%
Delta (exact):   -81.82%

RAG parent:      0.00%
Baseline parent: 81.82%
Delta (parent):  -81.82%

Retrieval hit:   0.00%


In [38]:
import json
print('RETRIEVAL DIAGNOSTICS: first 5 evaluated communities\n')
print('Chroma collection count:', collection.count())
for r in results[:5]:
    cid = int(r['community_id'])
    gt  = str(r['ground_truth'])
    hyde = str(r.get('hyde_query',''))
    print('\n' + '='*80)
    print(f"Community {cid} | ground_truth={gt}")
    print('HyDE snippet:', (hyde[:300] + '...') if hyde else '[empty]')
    try:
        retrieval = collection.query(query_texts=[hyde], n_results=TOP_K)
        docs = retrieval.get('documents', [[]])[0]
        metas = retrieval.get('metadatas', [[]])[0]
        retrieved_ids = [m.get('technique_id','') for m in metas]
        print('Retrieved IDs:', retrieved_ids)
        print('Ground-truth in retrieved_ids?', gt in retrieved_ids)
        # show first 3 metas and docs
        print('\nTop 3 metas:')
        for m in metas[:3]:
            print(' -', json.dumps(m, ensure_ascii=False))
        print('\nTop 3 docs (snippet):')
        for d in docs[:3]:
            s = (d[:700] + '...') if isinstance(d, str) and len(d) > 700 else d
            print(' -', s.replace('\n',' '))
        # check technique_id coverage
        non_empty = [m.get('technique_id') for m in metas if m.get('technique_id')]
        print('\nTechnique_id present in', len(non_empty), 'of', len(metas), 'retrieved metas')
    except Exception as e:
        print('  Retrieval query failed:', e)
print('\nDiagnostics complete.')


RETRIEVAL DIAGNOSTICS: first 5 evaluated communities

Chroma collection count: 703

Community 0 | ground_truth=T1110.001
HyDE snippet: The observed attack involves a DNS brute force attacker attempting to gain unauthorized access by guessing credentials on the DNS service running on port 53. This is followed by reconnaissance of the DNS authentication endpoint, indicating an attempt to gather more information about the target netwo...
Retrieved IDs: ['T1071.004', 'T1584', 'T1596.001', 'T1584.002', 'T1590.002', 'T1583.002', 'T1557', 'T1040', 'T1499', 'T1584.005']
Ground-truth in retrieved_ids? False

Top 3 metas:
 - {"name": "DNS", "technique_id": "T1071.004", "tactic": "Command And Control"}
 - {"name": "Compromise Infrastructure", "tactic": "Resource Development", "technique_id": "T1584"}
 - {"tactic": "Reconnaissance", "name": "DNS/Passive DNS", "technique_id": "T1596.001"}

Top 3 docs (snippet):
 - ID: T1071.004 Name: DNS Tactic: Command And Control Description: Adversaries may comm

In [39]:
import json
from sentence_transformers import util as sutil

print('EXTENDED RETRIEVAL DIAGNOSTICS: computing similarity against technique embeddings')

# Build technique texts and ids (same logic used when populating Chroma)
tech_ids = []
tech_texts = []
for tid, info in attck_techniques.items():
    tactic = (info.get('tactics') or [''])[0] if isinstance(info.get('tactics'), list) else (info.get('tactics') or '')
    text = f"ID: {tid}\nName: {info.get('name','')}\nTactic: {tactic}\nDescription: {info.get('description','')}"
    tech_ids.append(tid)
    tech_texts.append(text)

# Compute embeddings for technique descriptions (local, for diagnostics)
tech_embs = embedder.encode(tech_texts, convert_to_tensor=True)

diagnostics = []
TOP_CHECK = min(50, len(tech_ids))
for r in results[:5]:
    cid = int(r['community_id'])
    hyde = r.get('hyde_query','') or ''
    q_emb = embedder.encode(hyde, convert_to_tensor=True)
    sims = sutil.pytorch_cos_sim(q_emb, tech_embs)[0].cpu().tolist()
    topk_idx = sorted(range(len(sims)), key=lambda i: sims[i], reverse=True)[:TOP_CHECK]
    neighbors = []
    for i in topk_idx:
        neighbors.append({'technique_id': tech_ids[i], 'score': float(sims[i]), 'snippet': tech_texts[i][:300]})
    diagnostics.append({'community_id': cid, 'ground_truth': r.get('ground_truth'), 'hyde_query': hyde, 'top_neighbors': neighbors})

# Save diagnostics for offline inspection
out_path = RESULTS_DIR / 'retrieval_diagnostics.json'
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(diagnostics, f, indent=2)

print('Saved extended diagnostics to', out_path)
print(json.dumps(diagnostics[:1], indent=2))

EXTENDED RETRIEVAL DIAGNOSTICS: computing similarity against technique embeddings
Saved extended diagnostics to ../data/results/retrieval_diagnostics.json
[
  {
    "community_id": 0,
    "ground_truth": "T1110.001",
    "hyde_query": "The observed attack involves a DNS brute force attacker attempting to gain unauthorized access by guessing credentials on the DNS service running on port 53. This is followed by reconnaissance of the DNS authentication endpoint, indicating an attempt to gather more information about the target network's defenses and resources. Additionally, a command flooding botnet exfiltrates data from an HTTP web server, suggesting that sensitive or valuable data may be being stolen or compromised. An unknown actor exploits a vulnerability on an unspecified port, potentially compromising additional systems within the network.",
    "top_neighbors": [
      {
        "technique_id": "T1071.004",
        "score": 0.6414391994476318,
        "snippet": "ID: T1071.004\nNa